# «Навигатор по смыслу»: сравнение моделей семантического поиска по коду

**Задача:** Сравнить embedding-модели на маленьком датасете и ответить — какая работает лучше и почему.

## Теория

- **Эмбеддинг** — вектор фиксированной длины, в который модель превращает текст так, чтобы похожие по смыслу тексты оказывались близко друг к другу в этом пространстве.
- **Косинусное сходство** между векторами `a` и `b`:
  ```
  cos_sim(a, b) = (a · b) / (||a|| · ||b||)
  ```
  Диапазон [-1, 1]: 1 — максимально похожи, 0 — нет связи, -1 — противоположны. Если эмбеддинги заранее нормализовать (`normalize_embeddings=True`), косинусное сходство = обычное скалярное произведение.
- **Retrieval (поиск)**: один раз кодируем весь корпус ("индексация"), затем для каждого запроса кодируем его тем же способом и ищем `top-k` фрагментов корпуса с наибольшим косинусным сходством.
- **Precision@3**: доля вопросов, для которых эталонный ответ попал в top-3.
  ```
  Precision@3 = (число вопросов, где correct_chunk_id входит в top-3) / (всего вопросов)
  ```
- **MRR (Mean Reciprocal Rank)**: среднее обратное значение позиции эталона в выдаче (0, если не найден). Показывает не просто "попал в тройку", а "на каком месте".
  ```
  MRR = (1/N) * Σ (1 / rank_i)
  ```
- **t-SNE**: алгоритм проекции многомерных векторов (384/768 измерений) в 2D с сохранением локальной структуры — похожие по смыслу точки остаются близко и на плоском графике.

## Шаг 1. Загрузка данных и моделей

### 1.1 - загрузить датасет

В папке `dataset/` лежат три файла:
- `code_corpus.json` — список из 200 функций, у каждой поля `id`, `language`, `function_name`, `code`, `description`, `category`.
- `eval_questions.json` — список из 25 вопросов, поля `question_id`, `query`, `language`, `correct_chunk_id`.
- `categories.json` — `{"version": ..., "categories": [...]}`, у каждой категории `key`, `label`, `color`, `description`.


In [2]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path("..") / "dataset"

with open(DATA_DIR / "code_corpus.json", encoding="utf-8") as f:
    corpus = json.load(f)

with open(DATA_DIR / "eval_questions.json", encoding="utf-8") as f:
    questions = json.load(f)

with open(DATA_DIR / "categories.json", encoding="utf-8") as f:
    categories = json.load(f)["categories"]

category_color = {c["key"]: c["color"] for c in categories}
category_label = {c["key"]: c["label"] for c in categories}

print(f"Функций в корпусе: {len(corpus)}")
print(f"Вопросов для оценки: {len(questions)}")
print(f"Категорий: {len(categories)}")

Функций в корпусе: 200
Вопросов для оценки: 25
Категорий: 5


### 1.2 - выбрать и загрузить модели

### Выбор моделей

| Модель | Размер вектора | Почему она в сравнении |
|---|---|---|
| `paraphrase-multilingual-MiniLM-L12-v2` | 384 | Быстрая базовая многоязычная модель, рекомендована организаторами. Хорошая отправная точка по скорости. |
| `paraphrase-multilingual-mpnet-base-v2` | 768 | Более тяжёлая версия той же линейки (mpnet вместо MiniLM), обучена на большем числе языковых пар - ожидаем более точное понимание смысла за счёт большей ёмкости модели. |
| `intfloat/multilingual-e5-base` | 768 | В отличие от двух моделей выше, которые обучены на задаче *paraphrase/STS* (находить перефразировки), e5 обучена контрастивным способом специально на задаче *retrieval* — «запрос → релевантный документ» - и для этого использует асимметричные префиксы `query:` / `passage:`. На бенчмарке MTEB (retrieval-секция) e5 обычно превосходит paraphrase-модели, поэтому интересно проверить, даёт ли это преимущество на нашем небольшом датасете кода. |

Все три модели многоязычные

In [7]:
from sentence_transformers import SentenceTransformer, util

MODEL_NAMES = [
    "paraphrase-multilingual-MiniLM-L12-v2",
    "paraphrase-multilingual-mpnet-base-v2",
    "intfloat/multilingual-e5-base",
]

# e5 ожидает префиксы "query: " / "passage: " перед текстом - так модель отличает, что перед ней запрос, а что документ для поиска
# Поэтому заносим в отдельное множество, чтобы потом это учитывать
E5_MODELS = {"intfloat/multilingual-e5-base"}

models = {}
for name in MODEL_NAMES:
    print(f"Загрузка модели: {name} ...")
    t0 = time.time()
    models[name] = SentenceTransformer(name)
    print(f"  готово за {time.time() - t0:.1f} сек")

Загрузка модели: paraphrase-multilingual-MiniLM-L12-v2 ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  готово за 7.4 сек
Загрузка модели: paraphrase-multilingual-mpnet-base-v2 ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  готово за 7.2 сек
Загрузка модели: intfloat/multilingual-e5-base ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  готово за 7.3 сек


## Шаг 2. Генерация эмбеддингов и поиск

### Подготовка текста для кодирования в векторы

In [4]:
def corpus_text(item: dict) -> str:
    return f"{item['function_name']}\n{item['description']}\n\n{item['code']}"

def to_corpus_input(model_name: str, text: str) -> str:
    return f"passage: {text}" if model_name in E5_MODELS else text

def to_query_input(model_name: str, text: str) -> str:
    return f"query: {text}" if model_name in E5_MODELS else text

corpus_texts = [corpus_text(item) for item in corpus]
corpus_id_by_idx = [item["id"] for item in corpus]
corpus_category_by_idx = [item["category"] for item in corpus]

### 2.1 - закодировать корпус и вопросы

Для каждой модели нужно:
1. Закодировать все 200 текстов корпуса в векторы.
2. Закодировать все 25 запросов тем же способом.

In [5]:
corpus_embeddings = {}
question_embeddings = {}

for name, model in models.items():
    print(f"Кодирование корпуса моделью {name} ...")
    texts = [to_corpus_input(name, t) for t in corpus_texts]
    corpus_embeddings[name] = model.encode(
        texts, batch_size=32, normalize_embeddings=True, show_progress_bar=False
    )

    query_texts = [to_query_input(name, q["query"]) for q in questions]
    question_embeddings[name] = model.encode(
        query_texts, batch_size=32, normalize_embeddings=True, show_progress_bar=False
    )

print("Эмбеддинги готовы:")
for name in MODEL_NAMES:
    print(f"  {name}: corpus {corpus_embeddings[name].shape}, queries {question_embeddings[name].shape}")

Кодирование корпуса моделью paraphrase-multilingual-MiniLM-L12-v2 ...
Кодирование корпуса моделью paraphrase-multilingual-mpnet-base-v2 ...
Кодирование корпуса моделью intfloat/multilingual-e5-base ...
Эмбеддинги готовы:
  paraphrase-multilingual-MiniLM-L12-v2: corpus (200, 384), queries (25, 384)
  paraphrase-multilingual-mpnet-base-v2: corpus (200, 768), queries (25, 768)
  intfloat/multilingual-e5-base: corpus (200, 768), queries (25, 768)


### 2.2 - топ-3 по косинусному сходству

Для каждого вопроса и каждой модели найдити 3 функции корпуса с наибольшим косинусным сходством.

In [ ]:
def top_k_for_question(name: str, q_idx: int, k: int = 3):
    sims = util.cos_sim(question_embeddings[name][q_idx], corpus_embeddings[name])[0].numpy()
    top_idx = np.argsort(-sims)[:k]
    return [(corpus_id_by_idx[i], float(sims[i])) for i in top_idx]

search_results = {
    name: [top_k_for_question(name, i) for i in range(len(questions))]
    for name in MODEL_NAMES
}

# пример для первого вопроса
example_name = MODEL_NAMES[0]
q_idx = 0
print(f"Вопрос: {questions[q_idx]['query']!r} (эталон: {questions[q_idx]['correct_chunk_id']})")
print(f"Топ-3 ({example_name}):")
for chunk_id, score in search_results[example_name][q_idx]:
    print(f"  {chunk_id}  cos_sim={score:.3f}")


Вопрос: 'как проверить, истёк ли токен?' (эталон: func_001)
Топ-3 (paraphrase-multilingual-MiniLM-L12-v2):
  func_171  cos_sim=0.402
  func_071  cos_sim=0.386
  func_161  cos_sim=0.324
